# 03 - Agentic RAG Walkthrough

**Lecture goal:** show how Agentic RAG treats retrieval systems as tools. Instead of always running one fixed retriever, an agent plans, chooses a tool, inspects evidence, and decides whether to answer or try again.

**Use case:** the same AI-labs corpus from notebooks 01 and 02. The agent can choose between:

- **Hybrid RAG** for exact terms, acronyms, paper titles, and general semantic retrieval.
- **KG-RAG** for entity relationships and multi-hop questions.

**Notebook sections:**
1. Define tools - expose Hybrid RAG and KG-RAG as callable tools
2. Build the agent loop - plan, call tool, reflect, and synthesize
3. Trace a query end-to-end - inspect the decision path
4. Example Agentic RAG inputs and outputs - compare tool choices
5. Discussion - when Agentic RAG is worth the extra complexity

The saved outputs are lecture examples. The packaged `AgenticRagPipeline` in `src/` is currently a scaffold, so this notebook includes a small deterministic teaching trace to make the agent loop visible to students.


## 1. Define Tools - Hybrid RAG and KG-RAG

An agent needs tools. In this repo, the natural tools are the two retrieval systems built in the previous notebooks:

```text
hybrid_search(question) -> good for exact terms + semantic retrieval
kg_search(question)     -> good for entities + relationships
```

In a live run, these tools call the actual `HybridRagPipeline` and `KGRagPipeline` objects.


In [1]:
from rag_compare.common import load_corpus
from rag_compare.hybrid_rag import HybridRagPipeline
from rag_compare.kg_rag import KGRagPipeline

# In a live run, these tools call Qdrant, Neo4j, embeddings, and an LLM.
docs = load_corpus()
hybrid = HybridRagPipeline.build(docs)
kg = KGRagPipeline.build(docs)

tools = {
    "hybrid_search": hybrid.query,
    "kg_search": kg.query,
}

print(f"Loaded {len(docs)} source documents")
print("Registered tools:")
for name in tools:
    print(f"- {name}")


Loaded 18 source documents
Registered tools:
- hybrid_search
- kg_search


### What happened in section 1?

1. The corpus was loaded once.
2. The Hybrid RAG pipeline created dense and sparse retrieval views.
3. The KG-RAG pipeline created a graph-backed retrieval view.
4. The agent now has two tools it can choose from.

This is the main idea of Agentic RAG: retrieval is no longer a single fixed pipeline. It becomes a set of actions the agent can choose among.


## 2. Build the Agent Loop - Plan, Tool, Reflect, Synthesize

A typical Agentic RAG loop has four conceptual nodes:

```text
question
  -> plan: choose a tool or subquestion
  -> tool: call Hybrid RAG, KG-RAG, SQL, web search, etc.
  -> reflect: decide whether the evidence is enough
  -> synthesize: produce the final answer
```

The production version would use an LLM planner and LangGraph state machine. For lecture, the next cells use a small deterministic router so students can see the mechanics without hidden LLM behavior.


In [2]:
from dataclasses import dataclass, field

@dataclass
class TeachingAgentState:
    question: str
    evidence: list[str] = field(default_factory=list)
    trace: list[str] = field(default_factory=list)
    final_answer: str | None = None

TeachingAgentState(
    question="Which AI lab was founded by former OpenAI employees and develops Claude?"
)


TeachingAgentState(question='Which AI lab was founded by former OpenAI employees and develops Claude?', evidence=[], trace=[], final_answer=None)

In [3]:
def plan_tool(question: str, evidence: list[str]) -> str:
    """Small teaching router that stands in for an LLM planner."""
    q = question.lower()
    if any(term in q for term in ["founded", "former", "co-founded", "relationship", "develops"]):
        return "kg_search"
    if any(term in q for term in ["exact", "title", "acronym", "stand for", "year"]):
        return "hybrid_search"
    if evidence:
        return "synthesize"
    return "hybrid_search"

for question in [
    "Which AI lab was founded by former OpenAI employees and develops Claude?",
    "What is the exact title of the 2017 Transformer paper?",
    "What does RLHF stand for?",
]:
    print(f"Question: {question}")
    print(f"Planned tool: {plan_tool(question, [])}\n")


Question: Which AI lab was founded by former OpenAI employees and develops Claude?
Planned tool: kg_search

Question: What is the exact title of the 2017 Transformer paper?
Planned tool: hybrid_search

Question: What does RLHF stand for?
Planned tool: hybrid_search



### Planner intuition

The planner is answering: **which tool is most likely to retrieve the right evidence?**

| Query pattern | Better first tool | Why |
|---|---|---|
| `founded by`, `develops`, entity relationships | KG-RAG | Relationships are explicit graph edges |
| `exact title`, `acronym`, `year` | Hybrid RAG | BM25 helps with exact tokens |
| broad semantic question | Hybrid RAG | Dense retrieval helps with paraphrase |
| missing or weak evidence | loop / retry | Agent can reformulate or call another tool |


## 3. Trace a Query End-to-End - Decision Path and Evidence

Now we trace one multi-hop question. The important teaching artifact is not only the final answer. It is the path:

```text
question -> planned tool -> evidence -> reflection -> final answer
```


In [4]:
def lecture_agent_trace(question: str) -> TeachingAgentState:
    """A deterministic trace for lecture.

    This mirrors the intended Agentic RAG loop without requiring a live LLM planner
    in the notebook output. In production, the planner/reflect/synthesize steps
    would be LLM calls inside LangGraph.
    """
    state = TeachingAgentState(question=question)

    first_tool = plan_tool(state.question, state.evidence)
    state.trace.append(f"plan -> choose {first_tool}")

    if first_tool == "kg_search":
        state.evidence.append(
            "[kg] Anthropic is connected to former OpenAI employees as founders; "
            "Anthropic is also connected to Claude as a model it develops."
        )
    elif first_tool == "hybrid_search":
        state.evidence.append(
            "[hybrid] Retrieved chunks containing exact terms and semantic context for the question."
        )

    state.trace.append(f"tool -> collected {len(state.evidence)} evidence item")
    state.trace.append("reflect -> evidence is sufficient")
    state.trace.append("synthesize -> produce final answer")

    if "claude" in question.lower() and "founded" in question.lower():
        state.final_answer = "Anthropic."
    elif "title" in question.lower() and "transformer" in question.lower():
        state.final_answer = "Attention Is All You Need."
    elif "rlhf" in question.lower():
        state.final_answer = "Reinforcement Learning from Human Feedback."
    else:
        state.final_answer = "I need more evidence before answering."

    return state

trace = lecture_agent_trace(
    "Which AI lab was founded by former OpenAI employees and develops Claude?"
)

print("Input question:")
print(trace.question)
print("\nAgent trace:")
for step in trace.trace:
    print(f"- {step}")
print("\nEvidence:")
for item in trace.evidence:
    print(f"- {item}")
print("\nFinal answer:")
print(trace.final_answer)


Input question:
Which AI lab was founded by former OpenAI employees and develops Claude?

Agent trace:
- plan -> choose kg_search
- tool -> collected 1 evidence item
- reflect -> evidence is sufficient
- synthesize -> produce final answer

Evidence:
- [kg] Anthropic is connected to former OpenAI employees as founders; Anthropic is also connected to Claude as a model it develops.

Final answer:
Anthropic.


def lecture_agent_trace(question: str) -> TeachingAgentState:
    """A deterministic trace for lecture.

    This mirrors the intended Agentic RAG loop without requiring a live LLM planner
    in the notebook output. In production, the planner/reflect/synthesize steps
    would be LLM calls inside LangGraph.
    """
    state = TeachingAgentState(question=question)

    first_tool = plan_tool(state.question, state.evidence)
    state.trace.append(f"plan -> choose {first_tool}")

    if first_tool == "kg_search":
        state.evidence.append(
            "[kg] Anthropic is connected to former OpenAI employees as founders; "
            "Anthropic is also connected to Claude as a model it develops."
        )
    elif first_tool == "hybrid_search":
        state.evidence.append(
            "[hybrid] Retrieved chunks containing exact terms and semantic context for the question."
        )

    state.trace.append(f"tool -> collected {len(state.evidence)} evidence item")
    state.trace.append("reflect -> evidence is sufficient")
    state.trace.append("synthesize -> produce final answer")

    if "claude" in question.lower() and "founded" in question.lower():
        state.final_answer = "Anthropic."
    elif "title" in question.lower() and "transformer" in question.lower():
        state.final_answer = "Attention Is All You Need."
    elif "rlhf" in question.lower():
        state.final_answer = "Reinforcement Learning from Human Feedback."
    else:
        state.final_answer = "I need more evidence before answering."

    return state

trace = lecture_agent_trace(
    "Which AI lab was founded by former OpenAI employees and develops Claude?"
)

print("Input question:")
print(trace.question)
print("\nAgent trace:")
for step in trace.trace:
    print(f"- {step}")
print("\nEvidence:")
for item in trace.evidence:
    print(f"- {item}")
print("\nFinal answer:")
print(trace.final_answer)


## 4. Example Agentic RAG Inputs and Outputs

These examples show different tool choices for different questions. This is the educational payoff of Agentic RAG: one user interface, multiple retrieval strategies underneath.


In [5]:
questions = [
    "Which AI lab was founded by former OpenAI employees and develops Claude?",
    "What is the exact title of the 2017 Transformer paper?",
    "What does RLHF stand for?",
]

for question in questions:
    state = lecture_agent_trace(question)
    print("=" * 80)
    print(f"Input: {question}")
    print("Trace:")
    for step in state.trace:
        print(f"  {step}")
    print(f"Output: {state.final_answer}")


Input: Which AI lab was founded by former OpenAI employees and develops Claude?
Trace:
  plan -> choose kg_search
  tool -> collected 1 evidence item
  reflect -> evidence is sufficient
  synthesize -> produce final answer
Output: Anthropic.
Input: What is the exact title of the 2017 Transformer paper?
Trace:
  plan -> choose hybrid_search
  tool -> collected 1 evidence item
  reflect -> evidence is sufficient
  synthesize -> produce final answer
Output: Attention Is All You Need.
Input: What does RLHF stand for?
Trace:
  plan -> choose hybrid_search
  tool -> collected 1 evidence item
  reflect -> evidence is sufficient
  synthesize -> produce final answer
Output: Reinforcement Learning from Human Feedback.


### Input/output summary for lecture

| Stage | Example |
|---|---|
| User question | `Which AI lab was founded by former OpenAI employees and develops Claude?` |
| Planner decision | `kg_search` |
| Tool evidence | Anthropic is connected to former OpenAI employees and Claude |
| Reflection decision | Evidence is sufficient |
| Final answer | `Anthropic` |

For an exact-title question, the planner should choose `hybrid_search` first because exact terms and BM25 matter more.


### Where this maps to the repo code

The file `src/rag_compare/agentic_rag/pipeline.py` defines the intended LangGraph structure, but leaves `plan`, `reflect`, and `synthesize` as TODOs for students to implement.

That is why this notebook uses a teaching trace instead of calling `agent.query()` directly in the saved output. In a completed implementation, the same concepts would live inside LangGraph nodes.


In [6]:
# This repo's packaged AgenticRagPipeline is intentionally a scaffold.
# The production implementation should replace these teaching functions with
# LangGraph nodes: plan -> tool -> reflect -> synthesize.

from rag_compare.agentic_rag import AgenticRagPipeline

print(AgenticRagPipeline)
print("See src/rag_compare/agentic_rag/pipeline.py for the scaffold nodes.")


<class 'rag_compare.agentic_rag.pipeline.AgenticRagPipeline'>
See src/rag_compare/agentic_rag/pipeline.py for the scaffold nodes.


## 5. Discussion - When Agentic RAG Is Worth It

### What Agentic RAG is good at

- Questions that need different retrieval strategies.
- Heterogeneous sources: vector indexes, graphs, SQL, APIs, files, web search.
- Multi-step workflows where the first retrieval may not be enough.
- Debuggable traces showing which tools were used and why.

### What can go wrong

- More latency because the agent may make multiple LLM/tool calls.
- Higher cost because planning, reflection, and synthesis can each call an LLM.
- More nondeterminism because tool choice can vary.
- Loops can fail if stopping criteria are vague.
- Debugging is harder than a fixed pipeline unless traces are captured carefully.

### Questions for students

1. Why did the agent choose KG-RAG for the Anthropic/Claude question?
2. Why did it choose Hybrid RAG for the Transformer paper title?
3. What evidence would make the reflection step say "not enough"?
4. What guardrails would prevent an agent from looping forever?
5. When is Agentic RAG overkill compared with plain Hybrid RAG?
